# Lyric Engine - Kaggle Generation Workflow

This notebook handles cloning your codebase into Kaggle's `/kaggle/working` directory, and loading your PEFT model directly from an attached Kaggle Dataset in `/kaggle/input`.

## 1. One-Time Setup
Clone the repository to the writeable directory and ensure all pip packages are installed.

In [ ]:
import os

# 1. Move to Kaggle's permanent writable directory
os.chdir('/kaggle/working')

# 2. Clone the repository
if not os.path.exists('lyric-engine'):
    !git clone https://github.com/SMXFREEZE/lyric-engine

# 3. Move into the codebase
os.chdir('lyric-engine')

# 4. Install dependencies
!pip install -q transformers peft accelerate bitsandbytes trl datasets \
    lyricsgenius pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer

print("Setup Complete.")

## 2. Configuration & Checkpoint Prep
Validates that the specific Kaggle Dataset path actually contains your `adapter_config.json` file.

In [ ]:
import os
import torch

# ── Secrets (add via Kaggle sidebar → Add-ons → Secrets) ─────────────────────
# HF_TOKEN is required to download Mistral-7B (gated model on HuggingFace).
# Add it as a Kaggle Secret named HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets ✓")
except Exception:
    HF_TOKEN = ''
    print("No HF_TOKEN secret found — fine if using a public model")

# ── Checkpoint path (Kaggle dataset attached as input) ────────────────────────
# Kaggle datasets land at /kaggle/input/<dataset-slug>/
# The slug is the dataset name shown in the sidebar.
CKPT_DIR = '/kaggle/input/lyric-engine-trap-checkpoint'

# Auto-correct: if there's exactly one sub-folder (common when dataset was
# uploaded as a zip), descend into it automatically.
if os.path.isdir(CKPT_DIR):
    entries = os.listdir(CKPT_DIR)
    if len(entries) == 1 and os.path.isdir(os.path.join(CKPT_DIR, entries[0])):
        CKPT_DIR = os.path.join(CKPT_DIR, entries[0])

# Verify
if not os.path.exists(os.path.join(CKPT_DIR, 'adapter_config.json')):
    print(f"\nERROR: adapter_config.json not found in {CKPT_DIR}")
    print("Contents:", os.listdir(CKPT_DIR) if os.path.isdir(CKPT_DIR) else "directory missing")
    raise FileNotFoundError(f"Cannot find LoRA adapter at {CKPT_DIR}")

print(f"Checkpoint verified: {CKPT_DIR} ✓")

# ── Model / genre config ──────────────────────────────────────────────────────
BASE_MODEL = 'mistralai/Mistral-7B-v0.1'
GENRE      = 'trap'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_gpus = torch.cuda.device_count()
total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory / 1e9
    for i in range(n_gpus)
)

print(f"\nDevice     : {device.upper()}")
print(f"GPUs       : {n_gpus}")
print(f"Total VRAM : {total_vram:.1f} GB")
print(f"Base model : {BASE_MODEL}")
print(f"Genre      : {GENRE}")

## 3. Load Base Model and LoRA Adapter
Attaches your custom weights over the base 4-bit compressed Mistral model using the latest Transformers API.

In [ ]:
import os, sys, gc
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

sys.path.insert(0, '.')
from configs.genres import SPECIAL_TOKENS

# ── Free any leftover memory from earlier cells ───────────────────────────────
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {used:.1f} / {total:.1f} GB in use BEFORE load')

# ── 4-bit NF4 config ─────────────────────────────────────────────────────────
print("\nConfiguring 4-bit NF4 quantisation...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# ── Load base model (OOM-safe) ────────────────────────────────────────────────
# Root cause of the previous OOM:
#   • passing torch_dtype=bfloat16 alongside quantization_config forces the
#     full fp16 weight tensor to materialise on GPU *before* quantisation,
#     consuming all 14 GB on a single T4.
# Fixes applied here:
#   1. low_cpu_mem_usage=True  — quantises layer-by-layer from CPU; peak GPU
#      usage stays ~3.5 GB for Mistral-7B in 4-bit instead of ~14 GB.
#   2. No torch_dtype — bitsandbytes manages precision internally.
print(f"\nLoading {BASE_MODEL} in 4-bit (low_cpu_mem_usage=True)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,     # ← the fix that prevents OOM
    token=HF_TOKEN or None,
)
print("Base model loaded ✓")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
print("\nLoading tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
    print("  Found tokenizer in checkpoint ✓")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN or None)
    print("  Using base model tokenizer ✓")

tokenizer.pad_token = tokenizer.eos_token

new_tokens = [t for t in SPECIAL_TOKENS if t not in tokenizer.get_vocab()]
if new_tokens:
    print(f"  Adding {len(new_tokens)} Lyric Engine special tokens...")
    tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
    base_model.resize_token_embeddings(len(tokenizer))
print(f"  Vocab size: {len(tokenizer)}")

# ── Apply LoRA adapter ────────────────────────────────────────────────────────
print(f"\nApplying LoRA adapter from:\n  {CKPT_DIR}")
trained_model = PeftModel.from_pretrained(base_model, CKPT_DIR)
trained_model.eval()
print("LoRA attached ✓")

for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {used:.1f} / {total:.1f} GB in use AFTER load')

## 4. Full Lyric Generation Pipeline
Outputs lyrics with the loaded custom memory module.

In [ ]:
from src.inference.engine import LyricsEngine, SongMemory
import torch

engine = LyricsEngine(trained_model, tokenizer, device="cuda", beam_size=4)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f"\n======================================")
print(f"=== Generating full {GENRE.upper()} verse ===")
print(f"======================================\n")

with torch.no_grad():
    verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')

for line in verse:
    print(f"  {line}")